In [1]:
!pip install deepface opencv-python-headless tf-keras transformers pillow torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 35.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.9 MB/s eta 0:00:00


In [2]:
"""
Three-classifier GENDER detection on the 40-image manual-verification set.
Runs DeepFace + FairFace + CLIP on each image and outputs a CSV where each
row = one image with what all three classifiers said + whether they agree.
YOU then add a 'ground_truth' column by eye, and compute the two error types:
  (1) face missed but image is clearly a person
  (2) gender confused (male<->female)

SETUP:
  !pip install deepface opencv-python-headless tf-keras transformers pillow torch
  Input dataset: t2i_detection_40images (added). Verify path via os.listdir.
"""

import os, csv, warnings
warnings.filterwarnings("ignore")
import torch
from PIL import Image
from deepface import DeepFace
from transformers import pipeline, CLIPProcessor, CLIPModel

# ---- find the image root (walks /kaggle/input for the 40-image dataset) ----
INPUT_ROOT = "/kaggle/input"
IMG_ROOT = None
for dp, dirs, files in os.walk(INPUT_ROOT):
    if "40images" in dp.lower() and any(d.lower() in ("kndinsky","kandinsky","pixart","sd1.5","sdxl_turbo") for d in dirs):
        IMG_ROOT = dp; break
if IMG_ROOT is None:
    # fallback: the dataset folder itself
    for dp, dirs, files in os.walk(INPUT_ROOT):
        if "40images" in dp.lower():
            IMG_ROOT = dp; break
print("Image root:", IMG_ROOT)

OUT_CSV = "/kaggle/working/manual_verify_detection.csv"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- classifiers ----
print("Loading FairFace + CLIP...")
ff = pipeline("image-classification", model="dima806/fairface_gender_image_detection",
              device=0 if DEVICE=="cuda" else -1)
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
CLIP_LABELS = {"a photo of a man":"Man","a photo of a woman":"Woman"}

def norm(g):
    l=str(g).strip().lower()
    if l in ("male","man"): return "Man"
    if l in ("female","woman"): return "Woman"
    return ""

def clip_gender(pil):
    inp=clip_proc(text=list(CLIP_LABELS.keys()),images=pil,return_tensors="pt",padding=True).to(DEVICE)
    with torch.no_grad(): out=clip_model(**inp)
    p=out.logits_per_image.softmax(dim=1)[0]; i=int(p.argmax())
    return list(CLIP_LABELS.values())[i], float(p[i])

FIELDS=["model","image_file","face_detected",
        "df_gender","ff_gender","ff_score","clip_gender","clip_score",
        "all_agree","ground_truth","notes"]

rows=[]
for dp, dirs, files in os.walk(IMG_ROOT):
    # model name = the folder directly holding images (or its parent)
    model = os.path.basename(dp)
    for fn in sorted(files):
        if not fn.lower().endswith((".png",".jpg",".jpeg")): continue
        ipath=os.path.join(dp,fn)
        r={k:"" for k in FIELDS}
        r["model"]=model; r["image_file"]=fn; r["face_detected"]=False
        try:
            pil=Image.open(ipath).convert("RGB")
            # CLIP
            cg,cs=clip_gender(pil); r["clip_gender"]=cg; r["clip_score"]=round(cs,3)
            # DeepFace + crop
            crop=None
            try:
                res=DeepFace.analyze(img_path=ipath,actions=["gender"],
                                     detector_backend="retinaface",enforce_detection=True,silent=True)
                fa=res[0] if isinstance(res,list) else res
                r["face_detected"]=True; r["df_gender"]=norm(fa.get("dominant_gender",""))
                reg=fa.get("region",{}); x,y,w,h=reg.get("x",0),reg.get("y",0),reg.get("w",0),reg.get("h",0)
                if w>0 and h>0: crop=pil.crop((x,y,x+w,y+h))
            except ValueError:
                r["face_detected"]=False
            # FairFace on crop (or whole image if no crop)
            src=crop if crop is not None else pil
            fres=ff(src); top=max(fres,key=lambda z:z["score"])
            r["ff_gender"]=norm(top["label"]); r["ff_score"]=round(top["score"],3)
            # agreement
            gs=[g for g in [r["df_gender"],r["ff_gender"],r["clip_gender"]] if g]
            r["all_agree"]=(len(set(gs))==1) if len(gs)==3 else False
        except Exception as e:
            r["notes"]=str(e)[:100]
        rows.append(r)
        print(f"{model:14s} {fn:20s} DF={r['df_gender'] or '-':6s} FF={r['ff_gender'] or '-':6s} "
              f"CLIP={r['clip_gender'] or '-':6s} face={r['face_detected']} agree={r['all_agree']}")

with open(OUT_CSV,"w",newline="") as f:
    w=csv.DictWriter(f,fieldnames=FIELDS); w.writeheader(); w.writerows(rows)

print(f"\n{len(rows)} images processed. CSV: {OUT_CSV}")
print("\nNEXT: open the CSV, fill the 'ground_truth' column by eye (Male/Female/Unclear).")
print("Then the two error types are:")
print("  (1) MISSED FACE : ground_truth is a clear person but face_detected=False")
print("  (2) CONFUSED    : ground_truth != what a classifier said (male<->female)")

# quick agreement stats
agree=sum(1 for r in rows if r["all_agree"])
noface=sum(1 for r in rows if not r["face_detected"])
print(f"\nAll 3 agree: {agree}/{len(rows)} | Face missed: {noface}/{len(rows)}")

26-08-08 14:16:48 - Directory /root/.deepface has been created
26-08-08 14:16:48 - Directory /root/.deepface/weights has been created
Image root: /kaggle/input/datasets/shitichowdhury/t2i-40images/t2i_detection_40images
Loading FairFace + CLIP...


config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


... (18 outputs left out)

100%|██████████| 537M/537M [00:03<00:00, 159MB/s]  


pixart         img_10.png           DF=Woman  FF=Woman  CLIP=Woman  face=True agree=True
pixart         img_2.png            DF=Woman  FF=Woman  CLIP=Woman  face=True agree=True
pixart         img_3.png            DF=Man    FF=Man    CLIP=Man    face=True agree=True
pixart         img_4.png            DF=Man    FF=Man    CLIP=Man    face=True agree=True
pixart         img_5.png            DF=Man    FF=Man    CLIP=Man    face=True agree=True
pixart         img_6.png            DF=Man    FF=Man    CLIP=Man    face=True agree=True
pixart         img_7.png            DF=Woman  FF=Woman  CLIP=Woman  face=True agree=True
pixart         img_8.png            DF=Man    FF=Man    CLIP=Man    face=True agree=True
pixart         img_9.png            DF=Woman  FF=Woman  CLIP=Woman  face=True agree=True


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


sd1.5          img_1.png            DF=Man    FF=Man    CLIP=Man    face=True agree=True
sd1.5          img_10.png           DF=-      FF=Man    CLIP=Woman  face=False agree=False
sd1.5          img_2.png            DF=Man    FF=Man    CLIP=Man    face=True agree=True
sd1.5          img_3.png            DF=Man    FF=Man    CLIP=Man    face=True agree=True
sd1.5          img_4.png            DF=Man    FF=Man    CLIP=Man    face=True agree=True
sd1.5          img_5.png            DF=-      FF=Man    CLIP=Man    face=False agree=False
sd1.5          img_6.png            DF=Woman  FF=Woman  CLIP=Woman  face=True agree=True
sd1.5          img_7.png            DF=Woman  FF=Woman  CLIP=Woman  face=True agree=True
sd1.5          img_8.png            DF=Woman  FF=Woman  CLIP=Woman  face=True agree=True
sd1.5          img_9.png            DF=Woman  FF=Woman  CLIP=Woman  face=True agree=True
kndinsky       img_1.png            DF=Woman  FF=Woman  CLIP=Woman  face=True agree=True
kndinsky       im